# Inspect Childless Internal Nodes

This notebook identifies internal nodes that have no children within the dataset.
These nodes should be treated as leaf nodes since they have no descendants in our subset.

In [1]:
import pandas as pd
import pickle
from src.utils.ontology_utils import load_ontology, get_cell_info
from src.utils.paths import PROJECT_ROOT

In [2]:
# Load the cached ontology
cl = load_ontology()

# Load leaf and internal values from 10-24 data
data_folder = PROJECT_ROOT / "data" / "processed" / "10-24"

with open(data_folder / "2025-10-24_leaf_values.pkl", "rb") as f:
    leaf_values = pickle.load(f)

with open(data_folder / "2025-10-24_internal_values.pkl", "rb") as f:
    internal_values = pickle.load(f)

# All cell values in the dataset
all_cell_values = set(leaf_values + internal_values)

print(f"Loaded {len(leaf_values)} leaf nodes and {len(internal_values)} internal nodes")
print(f"Total nodes in dataset: {len(all_cell_values)}")

Loading cached ontology from /mnt/c/Users/zhaoj/projects/real_McCell/data/processed/ontology.pkl...
Ontology loaded successfully.
Loaded 23 leaf nodes and 57 internal nodes
Total nodes in dataset: 80


In [3]:
# Find internal nodes with NO CHILDREN in the dataset using parent_child_df
# parent_child_df: rows are all cells, columns are internal nodes
# Value = 1 if the column (internal node) is an ancestor of the row (cell)
# An internal node has children if any OTHER cell lists it as an ancestor

parent_child_df = pd.read_csv(
    data_folder / "2025-10-24_parent_child_df.csv",
    index_col=0
)

print(f"parent_child_df shape: {parent_child_df.shape}")
print(f"Rows (all cells): {len(parent_child_df.index)}")
print(f"Columns (internal nodes): {len(parent_child_df.columns)}")

# For each internal node (column), check if any cell OTHER than itself has it as ancestor
childless_internal = []
internal_with_children = []

for internal_node in parent_child_df.columns:
    col = parent_child_df[internal_node]
    # Exclude self from the check
    col_without_self = col.drop(internal_node, errors='ignore')
    # Count cells that have this internal node as ancestor
    num_children = (col_without_self == 1).sum()
    
    if num_children == 0:
        childless_internal.append(internal_node)
    else:
        internal_with_children.append(internal_node)

print(f"\nFound {len(childless_internal)} internal nodes with NO children in dataset")
print(f"Found {len(internal_with_children)} internal nodes WITH children in dataset")
print(f"Out of {len(parent_child_df.columns)} total internal nodes")

parent_child_df shape: (80, 57)
Rows (all cells): 80
Columns (internal nodes): 57

Found 22 internal nodes with NO children in dataset
Found 35 internal nodes WITH children in dataset
Out of 57 total internal nodes


In [4]:
# Alternative simpler approach: check using marginalization_df
# Internal nodes with no leaf descendants will have row sum = 0

marginalization_df = pd.read_csv(
    data_folder / "2025-10-24_marginalization_df.csv", 
    index_col=0
)

# Sum each row - nodes with sum=0 have no leaf descendants
row_sums = marginalization_df.sum(axis=1)
no_leaf_descendants = row_sums[row_sums == 0].index.tolist()

print(f"Internal nodes with NO leaf descendants: {len(no_leaf_descendants)}")
if no_leaf_descendants:
    print("\nThese nodes:")
    for node_id in no_leaf_descendants:
        print(f"  - {cl[node_id].name} ({node_id})")

Internal nodes with NO leaf descendants: 26

These nodes:
  - erythroid progenitor cell (CL:0000038)
  - common lymphoid progenitor (CL:0000051)
  - Kupffer cell (CL:0000091)
  - mast cell (CL:0000097)
  - erythrocyte (CL:0000232)
  - B cell (CL:0000236)
  - CD4-positive helper T cell (CL:0000492)
  - alveolar macrophage (CL:0000583)
  - neutrophil (CL:0000775)
  - plasmacytoid dendritic cell (CL:0000784)
  - mature B cell (CL:0000785)
  - memory B cell (CL:0000787)
  - naive B cell (CL:0000788)
  - gamma-delta T cell (CL:0000798)
  - double-positive, alpha-beta thymocyte (CL:0000809)
  - mature NK T cell (CL:0000814)
  - regulatory T cell (CL:0000815)
  - precursor B cell (CL:0000817)
  - hematopoietic multipotent progenitor cell (CL:0000837)
  - inflammatory macrophage (CL:0000863)
  - plasmablast (CL:0000980)
  - immature innate lymphoid cell (CL:0001082)
  - T follicular helper cell (CL:0002038)
  - intermediate monocyte (CL:0002393)
  - double negative thymocyte (CL:0002489)
  - l

In [5]:
# Display the childless internal nodes with details
if childless_internal:
    childless_data = []
    for node_id in childless_internal:
        childless_data.append({
            'cl_id': node_id,
            'name': cl[node_id].name
        })
    
    childless_df = pd.DataFrame(childless_data)
    childless_df = childless_df.sort_values('name').reset_index(drop=True)
    
    print("=" * 80)
    print("INTERNAL NODES WITH NO CHILDREN IN DATASET")
    print("These nodes are classified as 'internal' in the full ontology,")
    print("but should be treated as LEAF nodes in the training output.")
    print("=" * 80)
    print()
    
    # Display all rows
    with pd.option_context('display.max_rows', None):
        display(childless_df)
else:
    print("All internal nodes have at least one child in the dataset.")

INTERNAL NODES WITH NO CHILDREN IN DATASET
These nodes are classified as 'internal' in the full ontology,
but should be treated as LEAF nodes in the training output.



,cl_id,name
0,CL:0000091,Kupffer cell
1,CL:0002038,T follicular helper cell
2,CL:0000583,alveolar macrophage
3,CL:0000051,common lymphoid progenitor
4,CL:0002489,double negative thymocyte
5,CL:0000809,"double-positive, alpha-beta thymocyte"
6,CL:0000232,erythrocyte
7,CL:0000038,erythroid progenitor cell
8,CL:0000798,gamma-delta T cell
9,CL:0000837,hematopoietic multipotent progenitor cell


In [6]:
# Summary statistics
print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"Total leaf nodes (from ontology):     {len(leaf_values)}")
print(f"Total internal nodes (from ontology): {len(internal_values)}")
print(f"Childless internal nodes:             {len(childless_internal)}")
print()
print(f"Effective leaf nodes (should be):     {len(leaf_values) + len(childless_internal)}")
print(f"Effective internal nodes (should be): {len(internal_values) - len(childless_internal)}")
print()
print("If you want only one output at the end of the network,")
print(f"the output dimension should be: {len(leaf_values) + len(childless_internal)}")


SUMMARY
Total leaf nodes (from ontology):     23
Total internal nodes (from ontology): 57
Childless internal nodes:             22

Effective leaf nodes (should be):     45
Effective internal nodes (should be): 35

If you want only one output at the end of the network,
the output dimension should be: 45
